# 01. Run TRIBE v2

공식 `facebookresearch/tribev2` 추론 경로를 사용해 세그먼트별 cortical response를 생성한다.

- 입력: `outputs/prepared/{date}/segments.json`
- 처리: 각 세그먼트의 `tts_text`를 txt 파일로 저장 -> `model.get_events_dataframe(text_path=...)` -> `model.predict(events=df)`
- 출력: `outputs/raw/{date}-tribe-raw.npz`

주의:
- HuggingFace에서 `meta-llama/Llama-3.2-3B` access 승인과 로그인 토큰이 필요하다.
- 첫 실행은 모델/피처 추출기 다운로드로 시간이 오래 걸린다.

In [ ]:
!pip install -q uv
!uv pip install --system "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git"

In [ ]:
from huggingface_hub import login

# read 권한이 있는 HuggingFace token을 입력하세요.
login()

In [ ]:
from google.colab import drive
from pathlib import Path
import gc
import json
import numpy as np
import pandas as pd
import torch
from gtts import gTTS
from langdetect import detect
from tribev2.demo_utils import TribeModel, get_audio_and_text_events

drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive/tribe-v2-student-reaction')
PREPARED_DIR = ROOT / 'outputs' / 'prepared'
RAW_DIR = ROOT / 'outputs' / 'raw'
CACHE_DIR = ROOT / 'cache'
SEGMENT_TEXT_DIR = ROOT / 'segment_texts'

RAW_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
SEGMENT_TEXT_DIR.mkdir(parents=True, exist_ok=True)

PILOT_DATES = ['2026-02-02', '2026-02-09', '2026-02-24']
INFERENCE_MODE = 'audio_only'  # 'text_tts' | 'audio_only'
NUM_WORKERS = 0
BATCH_SIZE = 1

def build_audio_only_events(text: str, audio_path: Path):
    audio_path.parent.mkdir(parents=True, exist_ok=True)
    if not audio_path.exists():
        try:
            lang = detect(text)
        except Exception:
            lang = 'ko'
        if lang not in {'ko', 'en', 'ja', 'fr', 'de', 'es'}:
            lang = 'ko'
        gTTS(text, lang=lang).save(str(audio_path))

    audio_event = pd.DataFrame([
        {
            'type': 'Audio',
            'filepath': str(audio_path),
            'start': 0,
            'timeline': 'default',
            'subject': 'default',
        }
    ])
    return get_audio_and_text_events(audio_event, audio_only=True)

model = TribeModel.from_pretrained('facebook/tribev2', cache_folder=CACHE_DIR)
model.data.num_workers = NUM_WORKERS
model.data.batch_size = BATCH_SIZE
print('model ready', 'device=', model._model.device, 'num_workers=', model.data.num_workers, 'batch_size=', model.data.batch_size)

In [ ]:
def run_segment_prediction(date: str, segment: dict):
    segment_dir = SEGMENT_TEXT_DIR / date
    segment_dir.mkdir(parents=True, exist_ok=True)
    text_path = segment_dir / f"{segment['segment_id']}.txt"
    audio_path = segment_dir / f"{segment['segment_id']}.mp3"
    text_path.write_text(segment['tts_text'], encoding='utf-8')

    if INFERENCE_MODE == 'text_tts':
        events = model.get_events_dataframe(text_path=text_path)
    else:
        events = build_audio_only_events(segment['tts_text'], audio_path)

    preds, model_segments = model.predict(events=events)

    # 프론트에서는 세그먼트 단위 표현이 필요하므로 timestep 평균을 사용한다.
    mean_pred = preds.mean(axis=0).astype('float32')
    return {
        'segment_id': segment['segment_id'],
        'start_time': segment['start_time'],
        'end_time': segment['end_time'],
        'pred': mean_pred,
        'n_timesteps': int(preds.shape[0]),
        'n_vertices': int(preds.shape[1]),
        'events_preview': events[['type', 'start', 'duration']].head(12).to_dict(orient='records'),
    }


In [ ]:
for date in PILOT_DATES:
    segments = json.loads((PREPARED_DIR / date / 'segments.json').read_text(encoding='utf-8'))
    raw_path = RAW_DIR / f'{date}-tribe-raw.npz'
    meta_path = RAW_DIR / f'{date}-tribe-raw-meta.json'
    predicted_segments = []
    vectors = []

    if raw_path.exists() and meta_path.exists():
        existing = np.load(raw_path)['preds']
        vectors = [existing[idx] for idx in range(existing.shape[0])]
        predicted_segments = json.loads(meta_path.read_text(encoding='utf-8'))['segments']
        print('resume', date, 'from', len(predicted_segments), 'segments')

    for segment in segments[len(predicted_segments):]:
        result = run_segment_prediction(date, segment)
        vectors.append(result.pop('pred'))
        predicted_segments.append(result)
        print(date, result['segment_id'], 'timesteps=', result['n_timesteps'], 'vertices=', result['n_vertices'])

        preds = np.stack(vectors, axis=0)
        np.savez_compressed(
            raw_path,
            preds=preds,
            vertex_count=np.array([preds.shape[1]], dtype='int32'),
        )
        meta_path.write_text(
            json.dumps({'lecture_date': date, 'segments': predicted_segments}, ensure_ascii=False, indent=2),
            encoding='utf-8',
        )
        del preds
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    preds = np.stack(vectors, axis=0)
    np.savez_compressed(
        raw_path,
        preds=preds,
        vertex_count=np.array([preds.shape[1]], dtype='int32'),
    )
    meta_path.write_text(
        json.dumps({'lecture_date': date, 'segments': predicted_segments}, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
    print('saved raw output for', date, preds.shape)
